# 함수를 FastAPI 서버로 올리기



---
* `day17` 커널을 사용합니다.
* 17.2의 `FastAPI`, `GET`/`POST`/`DELETE` 를 활용해 python 함수를 서버로 실행해 보겠습니다.



In [2]:
!pip install fastapi uvicorn httpx -q

In [3]:
from pathlib import Path
WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)


WORKDIR : /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/17일차


---
## 노트 작성, 삭제 함수
* FastAPI 없이, 그냥 파이썬 함수로 추가·조회·삭제가 되는지 확인합니다.



In [4]:
NOTES: list[str] = []


def list_notes() -> dict: # 조회 함수
    return {'notes': NOTES, 'count': len(NOTES)}


def add_note(text: str) -> dict: # 노트 작성 함수
    NOTES.append(text)
    return {'ok': True, 'notes': list(NOTES)}


def delete_note(index: int) -> dict: # 노트 삭제 함수
    if index < 0 or index >= len(NOTES):
        return {'ok': False, 'error': f'index {index} 없음'}
    removed = NOTES.pop(index)
    return {'ok': True, 'removed': removed, 'notes': list(NOTES)}


print(add_note('FastAPI 공부'))
print(add_note('LangGraph Agent 서버'))
print(list_notes())
print(delete_note(0))
print(list_notes())


{'ok': True, 'notes': ['FastAPI 공부']}
{'ok': True, 'notes': ['FastAPI 공부', 'LangGraph Agent 서버']}
{'notes': ['FastAPI 공부', 'LangGraph Agent 서버'], 'count': 2}
{'ok': True, 'removed': 'FastAPI 공부', 'notes': ['LangGraph Agent 서버']}
{'notes': ['LangGraph Agent 서버'], 'count': 1}


---
## 2. 같은 함수를 엔드포인트에 매핑
* `@app.get` / `@app.post` / `@app.delete` 엔드포인트에 함수를 각각 연결합니다



In [5]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

NOTES.clear()  # 깨끗이 시작
app = FastAPI(title='Notes API')


@app.get('/notes')
def api_list_notes():
    return list_notes()


@app.post('/notes')
def api_add_note(text: str):
    return add_note(text)


@app.delete('/notes/{index}')
def api_delete_note(index: int):
    result = delete_note(index)
    if not result['ok']:
        raise HTTPException(status_code=404, detail=result['error'])
    return result


client = TestClient(app)
print([(sorted(r.methods), r.path) for r in app.routes if hasattr(r, 'methods')])

/Users/seorincho/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


[(['GET', 'HEAD'], '/openapi.json'), (['GET', 'HEAD'], '/docs'), (['GET', 'HEAD'], '/docs/oauth2-redirect'), (['GET', 'HEAD'], '/redoc'), (['GET'], '/notes'), (['POST'], '/notes'), (['DELETE'], '/notes/{index}')]


### TestClient로 작동 확인하기



In [6]:
print(client.get('/notes').json()) # 노트 조회하기 
print(client.post('/notes', params={'text': '첫 메모'}).json()) # 노트 작성
print(client.post('/notes', params={'text': '둘째 메모'}).json()) # 노트 작성
print(client.get('/notes').json()) # 노트 조회하기
print(client.delete('/notes/0').json()) # 0번 노트 삭제하기
print(client.get('/notes').json()) # 노트 조회하기

{'notes': [], 'count': 0}
{'ok': True, 'notes': ['첫 메모']}
{'ok': True, 'notes': ['첫 메모', '둘째 메모']}
{'notes': ['첫 메모', '둘째 메모'], 'count': 2}
{'ok': True, 'removed': '첫 메모', 'notes': ['둘째 메모']}
{'notes': ['둘째 메모'], 'count': 1}


---
## 3. 실제 서버로 띄우기



터미널 (`day17` 활성화 후):

```bash
cd 17일차_실습
uvicorn notes:app --reload
```

* 문서 UI: http://127.0.0.1:8000/docs
* 서버를 켠 뒤 아래 셀 실행



In [7]:
import httpx

BASE = 'http://127.0.0.1:8000'

# 서버가 안 떠 있으면 연결 오류가 납니다 → 위 uvicorn 먼저
with httpx.Client(base_url=BASE, timeout=5.0) as http:
    print('GET  /notes      →', http.get('/notes').json())
    print('POST /notes      →', http.post('/notes', params={'text': '서버에서 추가'}).json())
    print('POST /notes      →', http.post('/notes', params={'text': '하나 더'}).json())
    print('GET  /notes      →', http.get('/notes').json())
    print('DELETE /notes/0 →', http.delete('/notes/0').json())
    print('GET  /notes      →', http.get('/notes').json())



ConnectError: [Errno 61] Connection refused

---
## 4. 웹 페이지에서 보기
* API만 있으면 `/docs`로 테스트할 수 있지만, 실제 서비스는 **HTML UI**가 있는 경우가 많습니다.
* `GET /` 로 같은 서버에서 화면을 제공합니다.

서버 실행 중이면 브라우저에서 엽니다: http://127.0.0.1:8000/



---
## 정리
* **함수** → **엔드포인트** → **uvicorn** → (선택) **HTML** 로 브라우저에서 사용
* 다음(17.4)에서는 같은 방식으로 **LangGraph Agent** 를 `/chat` 뒤에 올립니다.
